In [1]:
!pip install plotly


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install nbformat


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv('../data/processed_smiley.xlsx')
df.info()

In [ ]:
import pandas as pd

filtered = df[df["category_group"] != "Preliminary Registrations"].copy()
filtered["is_severe"] = filtered["kontrol"].isin([3, 4])
city_avg = filtered["is_severe"].mean()

group_risk = (
    filtered.groupby("category_group")["is_severe"]
    .mean().div(city_avg).round(2)
    .to_frame("relative_risk").reset_index()
    .sort_values("relative_risk").reset_index(drop=True)
)

sub_risk = (
    filtered.groupby(["category_group", "category"])["is_severe"]
    .agg(severe_rate="mean", inspections="size").reset_index()
    .assign(relative_risk=lambda d: (d["severe_rate"] / city_avg).round(2))
    .query("inspections >= 100").reset_index(drop=True)
)

group_risk

In [ ]:
import re, json
import numpy as np

def inject(html, **kw):
    for k, v in kw.items():
        html = re.sub(rf'/\*{k}\*/.*?/\*END_{k}\*/',
                      lambda m, v=v: f'/*{k}*/{v}/*END_{k}*/', html, flags=re.DOTALL)
        html = re.sub(rf'<!--{k}-->.*?<!--/{k}-->',
                      lambda m, v=v: f'<!--{k}-->{v}<!--/{k}-->', html, flags=re.DOTALL)
    return html

class _NpEnc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return super().default(o)

def cr_color(rr):
    if rr >= 1.30:   return "rgba(239,68,68,0.75)"
    elif rr >= 1.0:  return "rgba(249,115,22,0.75)"
    elif rr <= 0.40: return "rgba(34,197,94,0.55)"
    else:            return "rgba(34,197,94,0.75)"

x_vals     = group_risk["relative_risk"].tolist()
y_vals     = group_risk["category_group"].tolist()
bar_colors = [cr_color(r) for r in x_vals]

def _hover(cat, rr):
    diff = abs(round((rr - 1.0) * 100))
    msg = f"<b>{diff}% {'more' if rr > 1 else 'less'} likely</b> than average"
    return f"<b>{cat}</b><br>Relative risk: <b>{rr:.2f}x</b><br>{msg}"

overview_js = json.dumps([{
    "type": "bar", "orientation": "h", "cliponaxis": False, "constraintext": "none",
    "x": x_vals, "y": y_vals,
    "marker": {"color": bar_colors, "line": {"width": 0}},
    "text": [f"  {r:.2f}x" for r in x_vals],
    "textposition": "outside",
    "textfont": {"size": 13, "family": "Manrope, Arial, sans-serif", "color": bar_colors},
    "customdata": [_hover(c, r) for c, r in zip(y_vals, x_vals)],
    "hovertemplate": "%{customdata}<extra></extra>",
}], cls=_NpEnc, ensure_ascii=False)

details_d = {}
for grp, sub in sub_risk.groupby("category_group"):
    sub = sub.sort_values("relative_risk")
    details_d[grp] = {
        "y": sub["category"].tolist(),
        "x": sub["relative_risk"].astype(float).tolist(),
        "colors": [cr_color(r) for r in sub["relative_risk"]],
        "hover": [
            f"<b>{row['category']}</b><br>Relative risk: <b>{row['relative_risk']:.2f}x</b>"
            f"<br>Inspections: {row['inspections']:,}"
            for _, row in sub.iterrows()
        ]
    }
details_js = json.dumps(details_d, cls=_NpEnc, ensure_ascii=False)

total_insp = len(filtered)
highest    = group_risk.iloc[-1]
lowest     = group_risk.iloc[0]
above      = group_risk[group_risk["relative_risk"] > 1.0]
second     = above.iloc[-2] if len(above) >= 2 else above.iloc[-1]

tmpl = open('../figures/category-risk.html').read()
out  = inject(tmpl,
    OVERVIEW_DATA = overview_js,
    DETAILS_DATA  = details_js,
    TOTAL_INSP    = f'{total_insp:,}',
    HIGHEST_BOX   = (f'{highest["category_group"]} are '
                     f'<span class="cr-hl red">{highest["relative_risk"]:.2f}&times;</span> more likely '
                     f'than the average business to receive a severe inspection outcome.'),
    SECOND_BOX    = (f'{second["category_group"]} businesses are '
                     f'<span class="cr-hl orange">{second["relative_risk"]:.2f}&times;</span> more likely than the average.'),
    LOWEST_BOX    = (f'{lowest["category_group"]} businesses are '
                     f'<span class="cr-hl green">{lowest["relative_risk"]:.2f}&times;</span> the average risk &mdash; the safest category.'),
)
open('../figures/category-risk.html', 'w').write(out)
open('../sections/category-risk.html', 'w').write(out)
print(f'Done. {total_insp:,} inspections | highest={highest["category_group"]} ({highest["relative_risk"]:.2f}x) | lowest={lowest["category_group"]} ({lowest["relative_risk"]:.2f}x)')